In [1]:
from langchain_community.document_loaders import (
    DirectoryLoader,
    PyPDFLoader,
    UnstructuredMarkdownLoader
)
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_text_splitters import Language
# Import the specific text splitter class from the LangChain text splitters module
from langchain_text_splitters import RecursiveCharacterTextSplitter


def load_pdfs(folder_path):
    print(f"Loading PDFs from {folder_path}...")
    loader = DirectoryLoader(folder_path, glob="**/*.pdf", loader_cls=PyPDFLoader)
    return loader.load()

def load_markdowns(folder_path):
    print(f"Loading Markdown files from {folder_path}...")
    loader = DirectoryLoader(folder_path, glob="**/*.md", loader_cls=UnstructuredMarkdownLoader)
    return loader.load()

def load_python_files(folder_path):
    print(f"Loading Python files from {folder_path}...")
    loader = GenericLoader.from_filesystem(
        path=folder_path,
        glob="**/*",
        suffixes=[".py"],
        parser=LanguageParser(language=Language.PYTHON, parser_threshold=500)
    )
    return loader.load()

# ==========================================
# Main Execution
# ==========================================
# Define your specific folder paths here
PDF_FOLDER = "../pdf_file"
MD_FOLDER = "../repo"
PY_FOLDER = "../repo"

# 1. Call each function with its specific folder
pdf_docs = load_pdfs(PDF_FOLDER)
md_docs = load_markdowns(MD_FOLDER)
py_docs = load_python_files(PY_FOLDER)

# 2. Combine all the loaded documents into one list
all_documents = pdf_docs + md_docs + py_docs

# 3. Print the final count
print("-" * 30)
print(f"Total documents loaded: {len(all_documents)}")

print (all_documents)



# Initialize the text splitter object and configure how it should divide the text
text_splitter = RecursiveCharacterTextSplitter(
    # Set the absolute maximum number of characters allowed in a single chunk
    chunk_size=1500,       
    
    # Set how many characters from the end of one chunk should be repeated at the start of the next
    chunk_overlap=200,     
    
    # *NEW*: Tell LangChain to track exactly where each chunk starts in the original document
    add_start_index=True
   
)

# Execute the splitting process on the loaded documents, returning a new list of smaller Document chunks
chunks = text_splitter.split_documents(all_documents)

# Print out an f-string that calculates and displays the total number of chunks generated
print(f"Split your document into {len(chunks)} chunks.")

# Print the string "Preview of chunk 1:" followed by the first 150 characters of the first chunk's content
print("Preview of chunk 1:", chunks[0].page_content[:150])

Loading PDFs from ../pdf_file...
Loading Markdown files from ../repo...
Loading Python files from ../repo...
------------------------------
Total documents loaded: 12
[Document(metadata={'producer': 'WeasyPrint 65.0', 'creator': 'PyPDF', 'creationdate': '', 'title': "MySQL Cheatsheet - CodeWithHarry's CheatSheets", 'source': '..\\pdf_file\\mysql_cheatsheet.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page_content="MySQL Cheatsheet\nWhat is a Database?\nA database is a structured collection of interrelated data stored together to serve\nmultiple applications.\nMySQL Elements\nLiterals\nLiterals refer to fixed data values:\nData Types\nMySQL provides several data types:\n17 -- Numeric literal\n'Harry'-- Text literal\n12.5 -- Real literal\n# Numeric Types\nTINYINT, SMALLINT, MEDIUMINT, INT, BIGINT\nFLOAT(M,D), DOUBLE(M,D), DECIMAL(M,D)\n# Date/Time Types\nDATE        -- YYYY-MM-DD\nDATETIME    -- YYYY-MM-DD HH:MM:SS\nTIME        -- HH:MM:SS"), Document(metadata={'producer': 'Wea

converting chunk to vector and saving it to chroma db


In [2]:
# Import the Chroma vector database from LangChain
from langchain_chroma import Chroma

# Import the Ollama embeddings class from the LangChain Ollama package
from langchain_ollama import OllamaEmbeddings

# Initialize the embedding model, telling it exactly which model you are running in Ollama
embeddings_model = OllamaEmbeddings(model="llama3.1")


# 2. Pass your original `chunks` (which STILL contain the metadata) into Chroma!
# Chroma will automatically extract the text, embed it, and save the text, vector, and metadata together.
vector_db = Chroma.from_documents(
    documents=chunks,           # Your 19 chunks (with text AND metadata)
    embedding=embeddings_model, # The model used to convert text to vectors
    persist_directory="./chroma_vector_db" # Saves the database to a folder on your computer!
)

print("Successfully embedded and saved all chunks WITH metadata into ChromaDB!")

Successfully embedded and saved all chunks WITH metadata into ChromaDB!
